# <font color="#418FDE" size="6.5" uppercase>**C: Diffusion Experiment**</font>
----

Last update: 20240603

By the end of this lecture, you will be able to:
* Develop a Diffusion-like generative model in the TensorFlow ecosystem.
* Build Diffusion model's functions & classes.
 * Near production-level functions & classes include [type hints](https://docs.python.org/3/library/typing.html) & formal descriptions with the [PEP 8](https://peps.python.org/pep-0008/) style implementation; we will see examples of them in this notebook.

## **1. Diffusion Models Overview Overview**

Diffusion models have emerged as a powerful class of generative models for image creation, offering high-quality & rsatile outputs. Here’s an overview of how they work & their significance in the field of image generation:

> **Basic Concept:**
- Diffusion models generate new data (like images) by gradually transforming a random pattern of noise into a structured image.
- This process is inspired by the physical process of diffusion, hence the name.
- The transformation occurs in a way that's somewhat reversed to how diffusion would naturally proceed—starting from randomness & ending with order.

> **Training Process:** The training of a diffusion model involves teaching the model to reverse a diffusion process:
   - **Forward Process:** This part involves gradually adding noise to an image until only random noise remains. This is done step-by-step over many iterations, where each step adds a little more noise.
   - **Reverse Process:** The model learns to reverse this noisy process to reconstruct the original image from the noise. It effectively learns to denoise images step by step.

> **Components of Diffusion Models:**
   - **Noise Schedule:** This is a predefined sequence that dictates how much noise to add at each step of the forward process.
   - **Neural Network:** Typically, a convolutional neural network is used to predict either the noise that was added at each step or directly predict the clean image from the noisy one.

> **Advantages**
   - **High-Quality Images:** Diffusion models are known for producing very high-quality & detailed images, comparable or sometimes superior to other generative models like GANs (Generative Adversarial Networks).
   - **Flexibility:** They can be conditioned on various inputs (like text or another image) to generate specific types of images, making them versatile for tasks like artistic image synthesis, photo editing, & more.
   - **Video generation:** Take a look at [Sora](https://openai.com/index/sora/) at OpenAI.



> **Applications**
   - **Art Generation:** They are used to create detailed & artistic images from textual descriptions.
   - **Photo Editing:** Useful in tasks like inpainting, where the model fills in missing parts of images, or super-resolution, enhancing the resolution of images.
   - **Simulation:** They can simulate how scenes develop under different conditions, useful in scientific & medical imaging.

> **Challenges**
   - **Computational Intensity:** The training & generation process can be computationally intensive due to the iterative nature of the reverse diffusion process.
   - **Control & Bias:** Ensuring the generated images do not propagate or amplify biases present in the training data is a challenge.

Diffusion models represent a significant advancement in the ability to generate complex images from simple inputs, & they continue to be a hot topic for research & development in the AI & computer vision communities.

##**2. A Simple Diffusion Experiment**

In this lecture, we use a U-Net architecture to create a near-production level diffusion model using TensorFlow ecosystem. The model here is adopted & modified from [here](https://tree.rocks/make-diffusion-model-from-scratch-easy-way-to-implement-quick-diffusion-model-e60d18fd0f2e).  

When setting up a diffusion model like a U-Net for training, creating the right configuration & environment is essential. This preparation involves specifying various parameters that govern the training process, model architecture, data handling, & output management. The configuration for a diffusion model typically includes settings such as the directory for training data, batch sizes, the number of training steps, the learning rate, & the noise scheduling parameters that are crucial for the model’s performance.

The configuration begins by defining a clear experiment name, which aids in systematically organizing outputs & logs. Specifying the training data directory ensures the model can access the required input images. Training parameters like batch size & the number of training steps determine the length & detail of the training phase. Additionally, specifying the noise schedule (how noise is added & removed during training) is vital for the effective learning of the model.

Operational settings in the configuration also include intervals for saving model checkpoints & plotting outputs, which are crucial for tracking training progress & allowing the training to be resumed from a specific point if interrupted. Settings for image sizes ensure that the model’s output matches the desired resolution, & purging old data ensures that each training session begins fresh, without the interference of outdated or irrelevant data files.

Advanced settings might include choosing to run the model in inference mode to generate images without further training & selecting the computational resource (CPU or GPU), which is vital for optimizing performance.

Utility functions are often utilized to initialize & set up the environment for these extensive configurations. These functions manage tasks like creating directories for saving outputs like samples, checkpoints, & model inferences, & they ensure these directories are prepared before training begins. They also dynamically adjust settings for model parameters based on specific needs, such as adapting the size & intensity of noise added during the training process, which directly affects the model’s effectiveness & performance. By automating these setup steps through utility functions, training a diffusion model becomes more efficient, reproducible, & adaptable to varying experimental conditions.

For example, in a project using a U-Net-based diffusion model, you might use a dataset of animal faces with images resized to lower dimensions to reduce computational demand. Similar to the GAN setup (previous lecture), the links to different sizes of this dataset can be provided for download, allowing for flexibility based on computational resources & desired output quality:

* https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_32.zip
* https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_64.zip
* https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_128.zip
* https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_256.zip
* https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_512.zip
* https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_1024.zip

In [ ]:
#@title Import Necessary Libraries
# Import necessary libraries for the project
import os
import glob
import copy
import math
import random
import time
import shutil
import requests
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import pandas as pd
from tqdm import tqdm
from matplotlib import pyplot as plt
from typing import *

# Google Colab library for mounting Google Drive
from google.colab import drive

# Mount Google Drive to access & store files
# Google Drive offers 15GB of free storage as of June 2024.
# Authentication will be required to access Google Drive (follow the prompts)
drive.mount('/content/drive')

# Explanation of Imports:
# - os, glob: for file & directory operations
# - copy: to perform deep & shallow copy operations
# - math, random: for mathematical operations & random number generation
# - time: for time-related tasks
# - shutil: for high-level file operations
# - requests: to make HTTP requests
# - tensorflow: for building & training ML models
# - numpy: for numerical operations
# - pandas: for data manipulation & analysis
# - tqdm: for displaying progress bars
# - matplotlib.pyplot: for plotting graphs
# - typing: for type hints

In [ ]:
#@title Class - Args for training

class Args:
    """
    Configuration settings for training a diffusion model.

    This class initializes & stores various configuration parameters that control the training process,
    model architecture, & operational settings such as data handling & output management.
    These settings are crucial for tailoring the behavior of the diffusion model to specific experimental requirements.
    """

    def __init__(self):
        """
        Initialize default settings for the diffusion model training session.
        """

        # General Mandatory Settings
        self.experiment_name: str = 'animal_faces'  # Name of the experiment for output organization
        self.project_type: str = 'diffusion'  # Type of the project
        self.project_parent_folder: str = '/content/drive/MyDrive'  # Base directory for project files

        # General Optional Settings
        self.batch_size: int = 64  # Number of samples per batch during training
        self.epochs: int = 500  # Total number of training epochs
        self.buffer_size: int = 1000  # Buffer size for dataset shuffling
        self.timesteps: int = 16  # Number of steps for noise reduction in generated images
        self.plot_intervals: int = 60  # Interval (in batches) to plot generator output
        self.checkpoint_intervals: int = 50  # Interval (in batches) to save model checkpoints
        self.image_size: int = 128  # Target output image size (width & height)
        self.folder_remove: bool = False  # Flag to clear old data in project directories at startup
        self.inference_mode: bool = False  # Run model in inference mode to generate images without training
        self.inference_num: int = 5  # Number of images to generate in inference mode
        self.gpu: bool = True  # Use GPU for training & inference, if available
        self.reduce_factor: int = 2  # Factor to reduce Conv layer filter sizes, affecting model capacity
        self.lr: float = 0.0008  # Learning rate for the optimizer
        self.beta1: float = 0.9  # First moment decay rate for the Adam optimizer
        self.beta2: float = 0.999  # Second moment decay rate for the Adam optimizer

        # Directory to store TensorFlow records
        self.dir_tfrecords: str = os.path.join(
            self.project_parent_folder, self.experiment_name, str(self.image_size), 'tfrecords'
        )

In [ ]:
#@title Counting Data in tfrecords
'''
Required Libraries:
import tensorflow as tf
'''

def count_records_in_tfrecord(tfrecord_path: str) -> int:
    """
    Count the number of records in a single TFRecord file.

    This function iterates through each record in the given TFRecord file & increments
    a counter to determine the total number of records.

    Args:
        tfrecord_path (str): The file path or URI to the TFRecord file. This can be a local
                             path or a remote path in a storage service like Google Cloud Storage
                             (indicated by a URI like gs://bucket_name/path_to_file).

    Returns:
        int: The total number of records in the specified TFRecord file.
    """
    count = 0
    # Initialize a TFRecordDataset to read from the specified file
    for _ in tf.data.TFRecordDataset(tfrecord_path):
        count += 1
    return count

def count_images_in_directory(directory_path: str) -> int:
    """
    Count the total number of images across all TFRecord files in a specified directory.

    This function first retrieves a list of all TFRecord files in the given directory,
    including support for remote directories in cloud storage. It then iterates through
    each TFRecord file, counting the records using `count_records_in_tfrecord`, and
    sums these counts to get the total number of images.

    Args:
        directory_path (str): The directory path containing TFRecord files. This can be a local
                              directory path or a remote directory path in a storage service
                              like Google Cloud Storage (e.g., "gs://bucket_name/path_to_directory/").

    Returns:
        int: The total number of records (images) across all TFRecord files in the directory.
    """
    total_count = 0
    # Use tf.io.gfile.glob to support both local & remote filesystems (e.g., GCS)
    tfrecord_files = tf.io.gfile.glob(os.path.join(directory_path, '*.tfrecord') )
    # Iterate through each file & count the records within
    for tfrecord_file in tfrecord_files:
        file_count = count_records_in_tfrecord(tfrecord_file)
        total_count += file_count
    return total_count


In [ ]:
#@title Custom Classes - Image Conversion
'''
Required libraries:
import tensorflow as tf
'''

class ImageConversion(tf.keras.layers.Layer):
    """
    A custom Keras layer for converting image data between different normalization ranges.

    This layer facilitates the conversion of image pixel values between two normalization
    schemes: [0, 1] & [-1, 1]. Depending on the `conversion_mode` specified during initialization,
    the layer can either normalize images (from [0, 1] to [-1, 1]) or denormalize them (from [-1, 1] to [0, 1]).

    Attributes:
        conversion_mode (int): Indicator of the conversion type to be applied. It supports:
            - -1 for [0, 1] to [-1, 1] conversion.
            - 0 for [-1, 1] to [0, 1] conversion.

    Methods:
        call(image): Applies the conversion to the image based on the `conversion_mode`.
        get_config(): Returns the configuration of the layer, including its `conversion_mode`.
    """

    def __init__(self, conversion_mode: int, **kwargs):
        """
        Initialize the ImageConversion layer with the specified mode of conversion.

        Args:
            conversion_mode (int): Mode of the conversion. -1 for normalizing to [-1, 1] and
                                    0 for denormalizing to [0, 1].
            **kwargs: Arbitrary keyword arguments for the base Layer class in Keras.
        """
        super().__init__(**kwargs)
        self.conversion_mode = conversion_mode

    def call(self, image: tf.Tensor) -> tf.Tensor:
        """
        Apply the specified image conversion when the layer is called.

        Args:
            image (tf.Tensor): Input tensor representing the image, with values expected to be in
                               the appropriate range based on the conversion mode.

        Returns:
            tf.Tensor: The converted image tensor.

        Raises:
            ValueError: If an unknown conversion mode is provided.
        """
        if self.conversion_mode == -1:
            return image * 2.0 - 1.0  # Convert from [0, 1] to [-1, 1]
        elif self.conversion_mode == 0:
            return image * 0.5 + 0.5  # Convert from [-1, 1] to [0, 1]
        else:
            # Instead of asserting, raise a ValueError for undefined conversion modes
            raise ValueError(f'Unknown conversion mode: {self.conversion_mode}')

    def get_config(self):
        """
        Overrides the default method to include the `conversion_mode` in the configuration.

        Returns:
            dict: Configuration dictionary containing the `conversion_mode` & base configuration.
        """
        config = super().get_config()
        config.update({
            'conversion_mode': self.conversion_mode
        })
        return config

In [ ]:
#@title Class - DiffusionModel
class DiffusionModel:
    def __init__(self, args):
        """
        Initialize the DiffusionModel with paths & settings derived from the provided configuration.

        Args:
            args (Args): Configuration object containing all required settings.
        """
        # Store configuration settings in the model
        self.args = args

        # Construct directory paths for experiments & data handling
        self.dir_experiment = os.path.join(args.project_parent_folder,
                                           args.experiment_name,
                                           str(args.image_size),
                                           args.project_type)
        self.dir_tfrecords = os.path.join(args.project_parent_folder,
                                          args.experiment_name,
                                          str(args.image_size),
                                          'tfrecords')
        self.dir_checkpoints = os.path.join(self.dir_experiment, 'checkpoints')
        self.dir_samples = os.path.join(self.dir_experiment, 'samples')
        self.dir_inferences = os.path.join(self.dir_experiment, 'inferences')

        # Conditionally clear directories if specified in configuration
        if args.folder_remove:
            self._clear_directories()

        # Create necessary directories for model operation
        self._create_directories()

        # Compute the number of batches per epoch based on the total number of images & batch size
        total_images = count_images_in_directory(self.dir_tfrecords)
        self.batch_num = int(total_images / args.batch_size)

        # Precompute a time-bar vector for timesteps, used in the diffusion process
        self.time_bar = 1 - np.linspace(0, 1.0, args.timesteps + 1)

        # Calculate the sizes of filters based on the reduction factor
        self.filters = int(512 / self.args.reduce_factor)
        self.mlp_filters = int(128 / self.args.reduce_factor)
        self.ts_filters = int(1.5 * 512 / self.args.reduce_factor)

        # List of image resolutions used in the model, calculated as powers of two
        self.resolutions = [2 ** i for i in range(2, int(np.log2(args.image_size)) + 1)]

        # Initialize the model & load weights if available
        self.model = self.create_model()
        self.model_load()

        # Configure the optimizer & loss function for model compilation
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=self.args.lr,
                                                  beta_1=self.args.beta1,
                                                  beta_2=self.args.beta2)
        self.loss_function = tf.keras.losses.MeanAbsoluteError()
        self.model.compile(loss=self.loss_function, optimizer=self.optimizer)

    # Why do some methods in Python start with a "_"?
    def _clear_directories(self):
        """Remove existing directories related to experiments, checkpoints, & outputs."""
        for directory in [self.dir_experiment, self.dir_tfrecords, self.dir_checkpoints,
                          self.dir_samples, self.dir_inferences]:
            if os.path.exists(directory):
                shutil.rmtree(directory)

    # Why do some methods in Python start with a "_"?
    def _create_directories(self):
        """Ensure all required directories exist for model training & output."""
        for directory in [self.dir_experiment, self.dir_tfrecords, self.dir_samples,
                          self.dir_checkpoints, self.dir_inferences]:
            os.makedirs(directory, exist_ok=True)

    def parse_function(self, example_proto: tf.Tensor) -> tf.Tensor:
        """
        Parse & process data from a TFRecord file.

        Args:
            example_proto (tf.Tensor): A tensor containing a serialized TFRecord example.

        Returns:
            tf.Tensor: A tensor containing a processed image.
        """
        # Define the features in the TFRecord that are to be extracted.
        feature_description = {
            'image_raw': tf.io.FixedLenFeature([], tf.string),
        }
        # Parse the input `tf.train.Example` proto using the dictionary above.
        example = tf.io.parse_single_example(example_proto, feature_description)
        # Decode the image, assume RGB channels.
        image = tf.io.decode_png(example['image_raw'], channels=3)
        # Convert the image to floating point values & normalize the image to the range [0, 1].
        image = tf.cast(image, tf.float32) / 255.0

        return image

    def load_dataset(self) -> tf.data.Dataset:
        """
        Load & parse the dataset from specified TFRecord files.

        Returns:
            tf.data.Dataset: A TensorFlow Dataset object containing processed images.
        """
        # Get all available tfrecords
        tfrecord_paths = tf.io.gfile.glob(os.path.join(self.dir_tfrecords, '*.tfrecord'))

        # Create a dataset from the file paths.
        dataset = tf.data.TFRecordDataset(tfrecord_paths)
        # Map parsing function across dataset elements with parallel processing.
        dataset = dataset.map(self.parse_function, num_parallel_calls=tf.data.experimental.AUTOTUNE)

        # # Shuffle the dataset using the buffer size specified in the configuration.
        # dataset = dataset.shuffle(self.opt['buffer_size'])

        # Batch the dataset with the specified batch size from the configuration.
        dataset = dataset.batch(self.args.batch_size)

        # Take the specified number of batches to ensure each epoch has a consistent number of batches.
        dataset = dataset.take(self.batch_num)

        # Prefetch the dataset to improve training efficiency.
        dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)

        # Return the fully prepared dataset.
        return dataset

    def generate_ts(self) -> np.array:
        """
        Generate random timestep indices for a batch of images within the model's defined timestep range.

        Returns:
            np.array: An array of random timesteps for each image in the batch.
        """
        # Generate random integers between 0 & the number of timesteps, one per batch item
        return np.random.randint(0, self.args.timesteps, size=self.args.batch_size)

    def forward_noise(self, x: np.array, t: np.array) -> (np.array, np.array):
        """
        Apply the forward noise process of the diffusion model to simulate the diffusion steps.

        Args:
            x (np.array): The current state of images, represented as a numpy array.
            t (np.array): Array of timestep indices at which to apply the noise.

        Returns:
            tuple[np.array, np.array]: A tuple of numpy arrays representing the images with noise applied for
                                       the current timestep `t` & the next timestep `t+1`.
        """
        # Retrieve scaling factors from precomputed time-bar at current & next timesteps
        a = self.time_bar[t]       # Get the current timestep scaling factor
        b = self.time_bar[t + 1]   # Get the next timestep scaling factor

        # Generate a noise array matching the shape of the input images
        noise = np.random.normal(size=x.shape)

        # Reshape scaling factors for broadcasting over the image dimensions
        a = a.reshape((-1, 1, 1, 1))
        b = b.reshape((-1, 1, 1, 1))

        # Apply the noise transformation for the current & next timesteps
        img_a = x * (1 - a) + noise * a  # Image for the current timestep
        img_b = x * (1 - b) + noise * b  # Predicted image for the next timestep

        return img_a, img_b

    def block(self, x_img: tf.Tensor, x_ts: tf.Tensor) -> tf.Tensor:
        """
        Perform a transformation block in the network that integrates spatial & temporal information.

        This method represents a single block of operations in a neural network, involving convolutions,
        activation functions, & interaction between image features & timestep embedding.

        Args:
            x_img (tf.Tensor): Input tensor for image features.
            x_ts (tf.Tensor): Input tensor for timestep features.

        Returns:
            tf.Tensor: Output tensor after applying convolutional operations & integrating time information.
        """
        # Apply a convolutional layer to the input image features
        x_parameter = layers.Conv2D(self.filters, kernel_size=3, padding='same')(x_img)
        x_parameter = layers.Activation('relu')(x_parameter)

        # Process timestep features through a dense layer, activate, & reshape for broadcasting
        time_parameter = layers.Dense(self.filters)(x_ts)
        time_parameter = layers.Activation('relu')(time_parameter)
        time_parameter = layers.Reshape((1, 1, self.filters))(time_parameter)

        # Element-wise multiplication of image features with time-based parameters to incorporate temporal effects
        x_parameter *= time_parameter

        # Apply another convolutional layer & add the processed time-parameterized features
        x_out = layers.Conv2D(self.filters, kernel_size=3, padding='same')(x_img)
        x_out += x_parameter  # Incorporate enhanced features into the main flow

        # Normalize & activate the output to ensure stability & non-linearity
        x_out = layers.LayerNormalization()(x_out)
        x_out = layers.Activation('relu')(x_out)

        return x_out

    def create_model(self) -> tf.keras.models.Model:
        """
        Construct the U-Net architecture model with integration of timestep information for the diffusion process.

        Returns:
            tf.keras.models.Model: The constructed U-Net model with input layers for images & timesteps.
        """
        # Input layer for images
        x_input = layers.Input(shape=(self.args.image_size, self.args.image_size, 3), name='x_input')
        # Placeholder for ImageConversion - not defined, assuming preprocessing operation
        x = ImageConversion(conversion_mode=-1)(x_input)

        # Input layer for timestep encoding
        x_ts_input = layers.Input(shape=(1,), name='x_ts_input')
        # Process timestep information through dense layer & normalization
        x_ts = layers.Dense(self.ts_filters)(x_ts_input)
        x_ts = layers.LayerNormalization()(x_ts)
        x_ts = layers.Activation('relu')(x_ts)

        # U-Net encoder: progressively downsample & increase feature dimensions
        x_encoder = {}
        for resolution in self.resolutions[::-1]:
            x = x_encoder[resolution] = self.block(x, x_ts)
            if resolution != 4:
                x = layers.MaxPool2D(2)(x)  # Downsample if not the lowest resolution

        # Flatten & merge image & timestep features for dense processing
        x = layers.Flatten()(x)
        x = layers.Concatenate()([x, x_ts])
        x = layers.Dense(self.filters)(x)
        x = layers.LayerNormalization()(x)
        x = layers.Activation('relu')(x)

        # Prepare for expansive path of U-Net
        x = layers.Dense(4 * 4 * self.mlp_filters)(x)
        x = layers.LayerNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Reshape((4, 4, self.mlp_filters))(x)

        # U-Net decoder: progressively upsample & decrease feature dimensions
        for resolution in self.resolutions:
            x = layers.Concatenate()([x, x_encoder[resolution]])  # Skip connection from encoder to decoder
            x = self.block(x, x_ts)
            if resolution != self.args.image_size:
                x = layers.UpSampling2D(2)(x)  # Upsample if not at final image size

        # Output layer: produce final image predictions
        x = layers.Conv2D(3, kernel_size=1, padding='same')(x)

        # Construct & return the model
        model = tf.keras.models.Model(inputs=[x_input, x_ts_input], outputs=x)
        return model

    def predict(self) -> np.array:
        """
        Generate images using the trained model by simulating the reverse diffusion process.

        Returns:
            np.array: An array containing the generated images.
        """
        # Initialize random noise as the starting point for image generation
        x = np.random.normal(size=(self.args.inference_num, self.args.image_size, self.args.image_size, 3))

        # Iteratively refine the images by predicting from random noise using the model across all timesteps
        for i in range(self.args.timesteps):
            t = i
            # Update the images based on the model prediction, using a constant timestep input across the batch
            x = self.model.predict([x, np.full(self.args.inference_num, t)], verbose=0)

        # Process & concatenate the final output to form a single image or batch of images
        return self.process_and_concat(x)

    def process_and_concat(self, img: np.array) -> np.array:
        """
        Post-process the predicted images & concatenate them for display or output.

        Args:
            img (np.array): The batch of images to process.

        Returns:
            np.array: The concatenated image array after processing.
        """
        # Normalize the image data to [0, 1] by adjusting based on the minimum & maximum values
        img = img - img.min()
        img = (img / img.max())
        img = img.astype(np.float32)

        # Concatenate images in the batch into a single array for easier visualization or further processing
        return np.concatenate([i for i in img], axis=0)

    def predict_step(self, num: int):
        """
        Generate & visualize the image generation process at specified timestep intervals.

        Args:
            num (int): The number of images to generate.
        """
        # Initialize a list to store images from certain timesteps
        xs = []
        # Start with random noise as the input
        x = np.random.normal(size=(num, self.args.image_size, self.args.image_size, 3))

        # Generate images step by step through all timesteps
        for t in range(self.args.timesteps):
            x = self.model.predict([x, np.full(num, t)], verbose=0)
            # Store images at intervals of 2 timesteps
            if (t+1) % 2 == 0:
                xs.append(self.process_and_concat(x))

        # Concat the images horizontally
        image = np.concatenate(xs, axis = 1)
        # Save the figure to a file
        filename = os.path.join(self.dir_inferences, f'step_{int(time.time())}.png')
        tf.keras.utils.save_img(filename, image)

    def train_on_batch(self, x_img: np.array) -> float:
        """
        Train the model on a single batch of images.

        Args:
            x_img (np.array): A batch of images to train on.

        Returns:
            float: The loss value for the batch.
        """
        x_ts = self.generate_ts()
        x_a, x_b = self.forward_noise(x_img, x_ts)
        loss = self.model.train_on_batch([x_a, x_ts], x_b)
        return loss

    def training(self):
        """
        Conduct the training process over multiple epochs & manage model evaluation & checkpoint saving.
        """
        dataset = self.load_dataset()

        for epoch in range(self.args.epochs):
            pbar = tqdm(dataset, total=self.batch_num, desc=f"Epoch {epoch + 1:04d} | lr: {float(self.model.optimizer.learning_rate):0.6f}", ncols=100, leave=False)
            running_loss = 0

            for batch, x_train in enumerate(pbar, start=1):
                loss = self.train_on_batch(x_train)
                running_loss += loss

                # Plot generated samples at specified intervals
                if (batch + 1) % self.args.plot_intervals == 0:
                    self.plot()

                # Save model checkpoints at specified intervals
                if (batch + 1) % self.args.checkpoint_intervals == 0:
                    self.model_save()

                # Update progress bar with the average loss
                pbar.set_postfix(loss=f"{running_loss / batch:0.8f}")

            # Reduce the learning rate for the next epoch
            self.model.optimizer.learning_rate = max(0.000001, self.model.optimizer.learning_rate * 0.99)

    def plot(self):
        """
        Generate & save an image using the current state of the model & manage image storage.
        """
        image = self.predict()
        filename = os.path.join(self.dir_samples, f'samples_{int(time.time())}.png')

        # Manage the storage of sample images by limiting the number of saved images
        previous_images = glob.glob(os.path.join(self.dir_samples, '*.png'))
        if len(previous_images) >= 10:
            for file_path in previous_images:
                try:
                    os.remove(file_path)
                except Exception as e:
                    print(f"Error removing {file_path}: {e}")

        tf.keras.utils.save_img(filename, image)

    def model_save(self):
        """
        Save the current model weights to a specified directory.
        """
        # Define the filename for saving the model weights
        filename = os.path.join(self.dir_checkpoints, 'diffusion.h5')
        # Save the model weights
        self.model.save_weights(filename)

    def model_load(self):
        """
        Load the model weights from the specified directory if they exist.
        """
        try:
            # Define the filename for loading the model weights
            filename = os.path.join(self.dir_checkpoints, 'diffusion.h5')
            if os.path.exists(filename):
                # Load the weights if the file exists
                self.model.load_weights(filename)
        except Exception as e:
            # Handle exceptions that may occur during model weight loading
            print(f"Failed to load model weights from {filename}: {e}")

    def generate_new_images(self):
        """
        Generate new images using the model & save them to the specified directory.
        """
        # Generate some step images
        for _ in range(self.args.inference_num):
            self.predict_step(1)

        # Ensure the model is loaded with the latest weights
        self.model_load()

        # Generate an image using the model's predict method
        images = self.predict().reshape(self.args.inference_num,
                                        self.args.image_size,
                                        self.args.image_size,
                                        3)
        for image in images:
            # Generate a unique filename based on the current timestamp
            name = f"{int(time.time() * 1e19):020d}.png"
            filename = os.path.join(self.dir_inferences, name)

            # Save the generated image
            tf.keras.utils.save_img(filename, image)


In [ ]:
#@title Main Run - Training
# Set TensorFlow logging level to reduce log noise
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Initialize configuration & model
args = Args()
my_obj = DiffusionModel(args)

# Check for the presence of TFRecord files
tfrecords_path = glob.glob(os.path.join(args.dir_tfrecords, '*'))
if len(tfrecords_path) == 0:
    # Download & unzip dataset if not present
    dataset_url = f"https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_{args.image_size}.zip"
    os.system(f"wget {dataset_url}")
    os.system(f"unzip animal_faces_{args.image_size}.zip -d {args.dir_tfrecords}")
    os.remove(f"animal_faces_{args.image_size}.zip")

    # Delay to ensure files have synced & are visible in Google Drive
    print("Waiting for Google Drive sync...")
    time.sleep(900)  # 15 minutes

    # Reboot the system to refresh the environment - Note: This will stop the script execution!
    print("Rebooting system to finalize setup...")
    os.system("sudo reboot")

# Display model architecture summary
my_obj.model.summary()

# Short delay for readability of output
time.sleep(5)

# Device configuration for training or inference
if args.gpu:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        device = gpus[0].name.replace('/physical_', '')
        device_context = tf.device(device)
else:
    cpus = tf.config.list_physical_devices('CPU')
    device = cpus[0].name.replace('/physical_', '')
    device_context = tf.device(device)

# Execute the model training or inference in the appropriate device context
with device_context:
    if args.inference_mode:
        my_obj.generate_new_images()
    else:
        my_obj.training()

In [ ]:
#@title Class - Args for Inference

class Args:
    """
    Configuration settings for using a diffusion model in inference.

    This class initializes & stores various configuration parameters that control the inference process,
    model architecture, & operational settings such as data handling & output management.
    These settings are crucial for tailoring the behavior of the diffusion model to specific experimental requirements.
    """

    def __init__(self):
        """
        Initialize default settings for the diffusion model training session.
        """

        # General Mandatory Settings
        self.experiment_name: str = 'animal_faces'  # Name of the experiment for output organization
        self.project_type: str = 'diffusion'  # Type of the project
        self.project_parent_folder: str = '/content/drive/MyDrive'  # Base directory for project files

        # General Optional Settings
        self.batch_size: int = 64  # Number of samples per batch during training
        self.epochs: int = 500  # Total number of training epochs
        self.buffer_size: int = 1000  # Buffer size for dataset shuffling
        self.timesteps: int = 16  # Number of steps for noise reduction in generated images
        self.plot_intervals: int = 60  # Interval (in batches) to plot generator output
        self.checkpoint_intervals: int = 50  # Interval (in batches) to save model checkpoints
        self.image_size: int = 128  # Target output image size (width & height)
        self.folder_remove: bool = False  # Flag to clear old data in project directories at startup
        self.inference_mode: bool = True  # Run model in inference mode to generate images without training
        self.inference_num: int = 5  # Number of images to generate in inference mode
        self.gpu: bool = False  # Use GPU for training & inference, if available
        self.reduce_factor: int = 2  # Factor to reduce Conv layer filter sizes, affecting model capacity
        self.lr: float = 0.0008  # Learning rate for the optimizer
        self.beta1: float = 0.9  # First moment decay rate for the Adam optimizer
        self.beta2: float = 0.999  # Second moment decay rate for the Adam optimizer

        # Directory to store TensorFlow records
        self.dir_tfrecords: str = os.path.join(
            self.project_parent_folder, self.experiment_name, str(self.image_size), 'tfrecords'
        )


In [ ]:
#@title Main Run - Inference
# Set TensorFlow logging level to reduce log noise
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Initialize configuration & model
args = Args()
my_obj = DiffusionModel(args)

# Check for the presence of TFRecord files
tfrecords_path = glob.glob(os.path.join(args.dir_tfrecords, '*'))
if len(tfrecords_path) == 0:
    # Download & unzip dataset if not present
    dataset_url = f"https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_{args.image_size}.zip"
    os.system(f"wget {dataset_url}")
    os.system(f"unzip animal_faces_{args.image_size}.zip -d {args.dir_tfrecords}")
    os.remove(f"animal_faces_{args.image_size}.zip")

    # Delay to ensure files have synced & are visible in Google Drive
    print("Waiting for Google Drive sync...")
    time.sleep(900)  # 15 minutes

    # Reboot the system to refresh the environment - Note: This will stop the script execution!
    print("Rebooting system to finalize setup...")
    os.system("sudo reboot")

# # Display model architecture summary
# my_obj.model.summary()

# # Short delay for readability of output
# time.sleep(5)

# Device configuration for training or inference
if args.gpu:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        device = gpus[0].name.replace('/physical_', '')
        device_context = tf.device(device)
else:
    cpus = tf.config.list_physical_devices('CPU')
    device = cpus[0].name.replace('/physical_', '')
    device_context = tf.device(device)

# Execute the model training or inference in the appropriate device context
with device_context:
    if args.inference_mode:
        my_obj.generate_new_images()
    else:
        my_obj.training()

# <font color="#418FDE" size="6.5" uppercase>**C: Diffusion Experiment**</font>
----

In this lecture, you learned to:
* Develop a Diffusion-like generative model in the TensorFlow ecosystem.
* Build Diffusion model's functions & classes.

In the next Module (Module 6), we will go over "Generative AI, Part 2."
